# Network Traffic Protocol & Threat Classifier
## Supervised Machine Learning on NSL-KDD Dataset

### Project & Viva Overview:
This notebook demonstrates the complete supervised learning workflow:
1. **Task A (Binary Threat Detection)**: Classify traffic as `Normal (0)` or `Attack (1)`.
2. **Task B (Multi-class Protocol Classification)**: Classify traffic into 8 protocol groups (`http`, `smtp`, `ftp`, `ftp_data`, `domain_u`, `telnet`, `daytime`, `other`).
3. **Zero Data Leakage Rule**: In Task B, the `service` column is strictly removed from feature inputs $X_B$.
4. **Models Evaluated**: Decision Tree (Entropy), Support Vector Machine (RBF Kernel), and Gaussian Naive Bayes.
5. **Scikit-learn Pipelines**: Preprocessing (`ColumnTransformer`) and classifiers are bundled together.

### 1. Import Libraries and Single-Source Preprocessing Utilities

In [ ]:
import os
import sys
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add root project folder to sys.path
sys.path.append(os.path.abspath('..'))

from ml.data_loader import (
    load_dataset,
    prepare_task_a,
    prepare_task_b,
    get_preprocessor,
    CATEGORICAL_TASK_A,
    NUMERICAL_TASK_A,
    CATEGORICAL_TASK_B,
    NUMERICAL_TASK_B,
    TARGET_PROTOCOLS
)

from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

### 2. Load NSL-KDD Dataset

In [ ]:
train_df = load_dataset('../data/KDDTrain+.txt')
test_df = load_dataset('../data/KDDTest+.txt')

print(f"Training set shape: {train_df.shape}")
print(f"Testing set shape:  {test_df.shape}")
train_df.head()

### 3. Explore Class Distributions
- Normal vs Attack (Task A)
- 8 Protocol Classes (Task B)

In [ ]:
X_train_A, y_train_A = prepare_task_a(train_df)
X_test_A, y_test_A = prepare_task_a(test_df)

X_train_B, y_train_B = prepare_task_b(train_df)
X_test_B, y_test_B = prepare_task_b(test_df)

print("Task A Target Distribution (Train):")
print(y_train_A.value_counts(normalize=True).apply(lambda v: f"{v:.2%}"))

print("\nTask B Target Distribution (Train):")
print(y_train_B.value_counts(normalize=True).apply(lambda v: f"{v:.2%}"))

print(f"\nVERIFICATION: Is 'service' in Task B features? -> {'service' in X_train_B.columns}")

### 4. Training Helper Function
Wraps model in a scikit-learn `Pipeline` with the preprocessing `ColumnTransformer`.

In [ ]:
def train_and_eval(name, model, preprocessor, X_tr, y_tr, X_te, y_te):
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    
    t0 = time.time()
    pipe.fit(X_tr, y_tr)
    train_time = round(time.time() - t0, 3)
    
    t1 = time.time()
    y_pred = pipe.predict(X_te)
    predict_time = round(time.time() - t1, 3)
    
    acc = accuracy_score(y_te, y_pred)
    f1 = f1_score(y_te, y_pred, average='weighted', zero_division=0)
    print(f"[{name}] Train Time: {train_time}s | Predict Time: {predict_time}s | Accuracy: {acc:.4f} | F1: {f1:.4f}")
    return pipe, {"accuracy": acc, "f1": f1, "train_time": train_time, "predict_time": predict_time}

### 5. Train Models for Task A (Threat Detection)
SVM is trained on 20,000 samples (`random_state=42`) as per specification.

In [ ]:
train_df_svm = train_df.sample(n=20000, random_state=42)
X_train_A_svm, y_train_A_svm = prepare_task_a(train_df_svm)

prep_A = get_preprocessor(CATEGORICAL_TASK_A, NUMERICAL_TASK_A)

# 1. Decision Tree (Entropy)
dt_pipe_a, dt_metrics_a = train_and_eval(
    "Task A - Decision Tree",
    DecisionTreeClassifier(criterion='entropy', max_depth=15, random_state=42),
    prep_A, X_train_A, y_train_A, X_test_A, y_test_A
)

# 2. SVM (RBF Kernel, 20k rows)
svm_pipe_a, svm_metrics_a = train_and_eval(
    "Task A - SVM (RBF)",
    CalibratedClassifierCV(estimator=SVC(kernel='rbf', random_state=42), cv=2),
    prep_A, X_train_A_svm, y_train_A_svm, X_test_A, y_test_A
)

# 3. Gaussian Naive Bayes
gnb_pipe_a, gnb_metrics_a = train_and_eval(
    "Task A - GaussianNB",
    GaussianNB(),
    prep_A, X_train_A, y_train_A, X_test_A, y_test_A
)

### 6. Train Models for Task B (Protocol Classification)
`service` column is strictly excluded from features.

In [ ]:
X_train_B_svm, y_train_B_svm = prepare_task_b(train_df_svm)
prep_B = get_preprocessor(CATEGORICAL_TASK_B, NUMERICAL_TASK_B)

# 1. Decision Tree (Entropy)
dt_pipe_b, dt_metrics_b = train_and_eval(
    "Task B - Decision Tree",
    DecisionTreeClassifier(criterion='entropy', max_depth=15, random_state=42),
    prep_B, X_train_B, y_train_B, X_test_B, y_test_B
)

# 2. SVM (RBF Kernel, 20k rows)
svm_pipe_b, svm_metrics_b = train_and_eval(
    "Task B - SVM (RBF)",
    CalibratedClassifierCV(estimator=SVC(kernel='rbf', random_state=42), cv=2),
    prep_B, X_train_B_svm, y_train_B_svm, X_test_B, y_test_B
)

# 3. Gaussian Naive Bayes
gnb_pipe_b, gnb_metrics_b = train_and_eval(
    "Task B - GaussianNB",
    GaussianNB(),
    prep_B, X_train_B, y_train_B, X_test_B, y_test_B
)

### 7. Feature Importance Analysis (Decision Tree)

In [ ]:
feature_names_a = dt_pipe_a.named_steps['preprocessor'].get_feature_names_out()
importances_a = dt_pipe_a.named_steps['classifier'].feature_importances_
top_indices_a = np.argsort(importances_a)[::-1][:15]

plt.figure(figsize=(10, 5))
plt.title('Top 15 Features for Threat Detection (Task A)')
clean_names = [feature_names_a[i].replace('cat__', '').replace('num__', '') for i in top_indices_a]
plt.barh(clean_names[::-1], importances_a[top_indices_a][::-1], color='#3b82f6')
plt.xlabel('Entropy-based Feature Importance')
plt.tight_layout()
plt.show()